# Mini vector store with SQLite + NumPy

This notebook shows how to:

1. Store numeric vectors in SQLite as **BLOBs** (`tobytes()`).
2. Read them back into NumPy arrays (`frombuffer` + the **same dtype** you stored).
3. Search by **Euclidean distance in Python**. Plain SQLite cannot do real vector math on blobs (`ORDER BY abs(vector - ?)` compares raw bytes, not numbers).

**Schema:** table `vectors` has two columns: `id` (integer) and `vector` (blob). A `SELECT *` row is `(id, vector)` — indexes `[0]` and `[1]` only.

**Gotcha:** `np.array([1.2, 2.5, ...])` is `float64`; `np.array([4, 5, 6])` is `int64`. Decode with the matching dtype, or store everything as `float64` in production.


In [2]:
# sqlite3 talks to a local .db file. numpy holds vectors and does math.
import sqlite3
import numpy as np


In [4]:
# Opens (or creates) vector-db.db in the working directory.
# The cursor is what we use to run SQL (CREATE / INSERT / SELECT).
conn = sqlite3.connect('vector-db.db')
cursor = conn.cursor()


In [8]:
# IF NOT EXISTS: safe to re-run this cell.
# id: auto-incrementing primary key.
# vector: BLOB = raw bytes. SQLite will not treat this as a numeric array.
cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS vectors (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        vector BLOB NOT NULL
    )
    """
)


In [10]:
# Sample embeddings. NumPy infers dtype from the values:
#   vect1 has decimals  -> float64 (8 bytes per number, length 4)
#   vect2 / vect3 are ints -> int64 (8 bytes per number, length 3)
# Mixed dtypes/lengths make search harder; a real store uses one dtype and one size.
vect1 = np.array([1.2, 2.5, 3.7, 0.8])
vect2 = np.array([4, 5, 6])
vect3 = np.array([7, 8, 9])


In [11]:
# Preview: tobytes() is the exact payload we store in the BLOB column.
# It is not human-readable; frombuffer() later turns it back into numbers.
vect1.tobytes()


b'333333\xf3?\x00\x00\x00\x00\x00\x00\x04@\x9a\x99\x99\x99\x99\x99\r@\x9a\x99\x99\x99\x99\x99\xe9?'

In [13]:
# Insert vect1. "?" is bound from the tuple below (never concatenate SQL strings).
# sqlite3.Binary marks the payload as a BLOB. id is filled in automatically.
cursor.execute(
    """
    INSERT INTO vectors (vector) VALUES (?)
    """,
    (sqlite3.Binary(vect1.tobytes()),),
)


In [14]:
# Same insert for vect2 (int64 bytes, 3 elements).
cursor.execute(
    """
    INSERT INTO vectors (vector) VALUES (?)
    """,
    (sqlite3.Binary(vect2.tobytes()),),
)


In [15]:
# Same insert for vect3.
cursor.execute(
    """
    INSERT INTO vectors (vector) VALUES (?)
    """,
    (sqlite3.Binary(vect3.tobytes()),),
)


In [17]:
# SELECT * returns every column, in table order: (id, vector).
# fetchall() loads all matching rows into a Python list.
# Re-running INSERT cells will add duplicate rows; this SELECT will show all of them.
cursor.execute("SELECT * FROM vectors")
rows = cursor.fetchall()


In [33]:
# Decode the first row only.
# rows[0][0] is id; rows[0][1] is the blob. There is no index [2].
# frombuffer (lowercase b) reinterprets bytes. dtype MUST match how the array was stored.
vector = np.frombuffer(rows[0][1], dtype=np.float64)
vector


array([1.2, 2.5, 3.7, 0.8])

In [34]:
# rows already has every record from SELECT * (fetchall).
# Each row is (id, vector_blob). Loop instead of rows[0] only.
#
# vect1 was float64; vect2 and vect3 were integer arrays (int64).
# There is no single dtype that decodes both correctly.
for row_id, blob in rows:
    if row_id == 1:
        vec = np.frombuffer(blob, dtype=np.float64)
    else:
        vec = np.frombuffer(blob, dtype=np.int64)
    print(row_id, vec)

1 [1.2 2.5 3.7 0.8]
2 [4 5 6]
3 [7 8 9]


In [ ]:
# Query embedding: "find stored vectors close to this one".
# It matches vect1, so distance to row 1 should be 0.
query_vector = np.array([1.2, 2.5, 3.7, 0.8])


In [39]:
# Load all (id, blob) pairs for search in the next cell.
# Do not ORDER BY abs(vector - ?) — SQLite subtracts blobs as bytes,
# not as arrays, so "nearest neighbor" in SQL here is meaningless.
cursor.execute("SELECT id, vector FROM vectors")
rows = cursor.fetchall()


In [44]:
# Euclidean nearest-neighbor in Python (what a vector DB does internally).
# np.linalg.norm(a - b) = sqrt(sum of squared differences). Smaller = closer.
query = np.asarray(query_vector, dtype=np.float64)
np.set_printoptions(suppress=True, precision=4)  # avoid 1.2e+00 style output

def decode_blob(blob):
    # 4 float64s = 32 bytes (query.nbytes). 3 int64s = 24 bytes.
    if len(blob) == query.nbytes:
        return np.frombuffer(blob, dtype=np.float64).copy()
    return np.frombuffer(blob, dtype=np.int64).astype(np.float64)

def pad_to(a, b):
    # Distance needs equal length. Pad the shorter vector with zeros.
    # Production embeddings should all share one dimension so this is unnecessary.
    n = max(len(a), len(b))
    left = np.zeros(n, dtype=np.float64)
    right = np.zeros(n, dtype=np.float64)
    left[: len(a)] = a
    right[: len(b)] = b
    return left, right

for row_id, blob in rows:
    vec = decode_blob(blob)
    q, v = pad_to(query, vec)
    dist = float(np.linalg.norm(v - q))
    print(f"id={row_id}, distance={dist:.4f}")
    print(vec)
    print()


id=1, distance=0.0000
[1.2 2.5 3.7 0.8]

id=2, distance=4.4744
[4. 5. 6.]

id=3, distance=9.6239
[7. 8. 9.]

